# Media Sources Recognition with impresso-pipelines Package

<a target="_blank" href="https://colab.research.google.com/github/impresso/impresso-datalab-notebooks/blob/main/annotate/mediasources_pipeline_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

If something doesn't work, you can [report a problem](https://github.com/impresso/impresso-datalab-notebooks/blob/main/reporting-problems.md).

## What is this notebook about?

This notebook introduces the Media Sources component of the [impresso-pipelines](https://pypi.org/project/impresso-pipelines/) Python package. The broader goal of the Impresso pipelines is to make the internal data processing workflows of the Impresso project reusable and accessible to others. Researchers, developers and digital humanities practitioners can apply these processing steps to their own text collections.

In this notebook, we focus on the [mediasources](https://github.com/impresso/impresso-pipelines/tree/main/impresso_pipelines/mediasources) subpackage. It identifies mentions of **press agencies and radio stations** in newspaper text and links known sources to Wikidata. You will learn how to run the pipeline, locate mentions in the original text, and interpret and filter the results.



## Why is this useful?

Historical newspapers often name press agencies and radio stations when reporting news. These references offer clues about the sources newspapers cited and how they presented information to their readers. Finding them manually across thousands of articles would be time-consuming. The pipeline helps you identify relevant passages and turn those references into data you can compare.

For example, you could use the annotations to investigate:

- **Changes over time:** Did a newspaper begin citing radio broadcasts more often during a particular period?
- **Differences between newspapers:** Which papers frequently mentioned the same agencies, and which cited different sources?
- **Coverage of an event:** Which agencies or stations were cited in reports about a war, election or diplomatic crisis?
- **Sources across languages:** Where does the same agency appear in French-, German- or English-language reporting?

The results can help you select articles for close reading or build a starting dataset for studying patterns of source citation. Comparisons should account for differences in collection size, coverage and recognition quality.

A mention tells us that a source is named in the text. It does not by itself establish that the source produced the article or verify the information attributed to it.

## How it works

Text → Tokenization → Media source recognition → Filtering → Mentions and source summary

- **Initialize**: Load the model and tokenizer from Hugging Face.
- **Recognize**: Identify words that name a press agency or radio station and group them into a source mention, such as ‘BBC World Service’. Long documents are processed in overlapping chunks.
- **Filter**: Optionally remove mentions below a confidence threshold or sources founded after the supplied publication year.
- **Return results**: Keep the original character positions, attach known Wikidata identifiers, and summarize each source by its highest-scoring mention.

## Technical background

The default model is [mmBERT Impresso Media Sources NER](https://huggingface.co/impresso-project/mmbert-impresso-mediasources-ner), a multilingual token-classification model based on ModernBERT. It uses the surrounding text to distinguish source names from other words. Internally, BIO labels mark the beginning (`B`), continuation (`I`) or absence (`O`) of a mention. The pipeline decodes these labels and maps them back to the original text.

You do not need to implement these steps yourself. The examples below use the pipeline's public interface.

## What will you learn?

In this notebook, you will:

- Identify press agencies and radio stations in a sample text.
- Interpret mention details, confidence scores and the source summary.
- Extract a mention using its character positions and look up its Wikidata identifier.
- Adjust `min_score` and use publication dates to filter results.
- Process several documents together, including documents with different dates.
- Inspect diagnostics for token labels and long-document processing.
- Recognize limitations when working with OCR errors, abbreviations and historical dates.

## Useful resources

- [Impresso Pipelines package](https://github.com/impresso/impresso-pipelines): source code and installation instructions.
- [Media Sources pipeline documentation](https://github.com/impresso/impresso-pipelines/blob/main/README_mediasources.md): input, output and configuration details.
- [Media Sources model](https://huggingface.co/impresso-project/mmbert-impresso-mediasources-ner): model information and files.

## Prerequisites

Start by installing `impresso-pipelines` with the `mediasources` extra. This example uses package version `0.6.1.2` and model revision `v2.0.0`. Use Python 3.12, as in this repository's shared setup.

The model is downloaded from Hugging Face when you initialize the pipeline for the first time. An internet connection and enough memory to load the model are needed (~1.5 GB). Subsequent initializations can reuse the cached files. An Impresso API account is not required.

The pipeline selects an available accelerator (CUDA or Apple Silicon MPS), falling back to CPU. CPU execution can be slower. You can select CPU explicitly with `device="cpu"`.

In [ ]:
%pip install "impresso-pipelines[mediasources]"

> After installation, you may need to restart the kernel. In Google Colab, go to *Runtime* and select *Restart session*, then continue with the imports below.


## Basic Usage

Start by importing the pipeline. Initialize it once and reuse it for the examples below. Please be patient during the first model download.

In [ ]:
from impresso_pipelines.mediasources import MediaSourcesPipeline

mediasources_pipeline = MediaSourcesPipeline(
    model="impresso-project/mmbert-impresso-mediasources-ner",
    revision="v2.0.0",
)

Once the pipeline is initialized, provide the text you would like to annotate. This short example combines French, German and English sentences and repeats Reuters so that we can compare individual mentions with the source summary.

All texts in this notebook are **written for demonstration**. They are not quotations from historical newspapers. Exact spans and confidence scores depend on the model's predictions; inspect your own output rather than assuming every name will be detected.

In [ ]:
text = """Reuters reported the decision from London. The BBC World Service provided an update.
Selon l'Agence Havas, les discussions se poursuivent à Paris.
Die Deutsche Presse-Agentur (dpa) meldet weitere Gespräche.
Reuters later published a second report."""

result = mediasources_pipeline(text)
result

**Interpretation of the result:**

The pipeline returns a dictionary with two lists:

- `entities`: every detected mention. If Reuters is recognized twice, both occurrences appear here.
- `summary`: one entry per normalized source label, sorted by its highest mention score. This is a source list, not a count of mentions.

Each entry in `entities` contains:

| Field | Meaning |
| --- | --- |
| `surface` | The exact substring recognized in the input text. |
| `label` | A normalized source label, such as `org.ent.pressagency.reuters` or `org.ent.radiostation.bbc`. |
| `wkdata_qid` | A Wikidata identifier when a mapping is available; otherwise `None`. |
| `start_year` | The recorded start year for the source, when available. |
| `start`, `stop` | Character positions in the original text: start is included, stop is excluded. |
| `score` | The mean token confidence for this mention, between 0 and 1. |

Each `summary` entry contains `uid` (the normalized label), `wkdata_qid`, and the highest mention `score` for that label. A high score means the model is confident about its prediction, but it can still identify the wrong source or capture only part of its name.

In [ ]:
# Inspect all mentions and the separate list of unique sources.
for entity in result["entities"]:
    print(entity["surface"], entity["label"], round(entity["score"], 3))

print("\nSource summary:")
result["summary"]

### Locate mentions and look up sources

Character positions let you recover a mention directly from the original text. They count from zero, and `stop` points to the first character after the mention.

A Wikidata identifier is not itself a URL. We can use it to build a link. Some known sources have no mapping yet, so check for `None` before building the link.

In [ ]:
for entity in result["entities"]:
    surface = text[entity["start"]:entity["stop"]]
    qid = entity["wkdata_qid"]
    link = f"https://www.wikidata.org/wiki/{qid}" if qid else "No Wikidata mapping"
    print(f"{surface!r} [{entity['start']}:{entity['stop']}] → {link}")

## Advanced Usage

The following options help you control the output and understand the processing:

- `min_score`: remove mentions below a confidence threshold, for example `0.90`. By default, no additional score threshold is applied.
- `publication_date`: the date of the document you are annotating, supplied by you.
- `filter_anachronistic`: when set to `True`, uses the year from `publication_date` to remove detected sources whose recorded start year is later than the document year. For example, it removes the BBC (start year 1922) from results for a document dated 1920. Supplying a date alone does not enable filtering, and enabling the filter without a date has no effect. The filter is disabled by default.
- `batch_size`: the number of model chunks processed together. It can be supplied during initialization or for one call.
- `diagnostics`: add original text, tokens, token positions, BIO labels, token scores and chunking information.

These options belong to `MediaSourcesPipeline`.

**Example 1:** `min_score`

Set a threshold to keep only mentions whose score is at least `0.90`. This filters the decoded mentions and then rebuilds the summary.

In [ ]:
filtered_result = mediasources_pipeline(text, min_score=0.90)
print("Before filtering:", len(result["entities"]), "mentions")
print("After filtering:", len(filtered_result["entities"]), "mentions")
filtered_result

A higher threshold can reduce low-confidence predictions, but it may also remove correct mentions. A lower threshold cannot recover a source the model did not recognize. Choose a threshold by inspecting examples from your own collection.

**Example 2:** `publication_date` and `filter_anachronistic`

Imagine the following text has been assigned a publication year of 1920. The pipeline's mapping records Reuters as starting in 1851 and the BBC in 1922. If both are detected, the date filter keeps Reuters and removes the BBC.

This is a deliberately inconsistent example to show the filter. It does not represent an actual newspaper extract.

In [ ]:
dated_text = "Reuters reported the news. The BBC World Service provided an update."

without_date_filter = mediasources_pipeline(dated_text)
with_date_filter = mediasources_pipeline(
    dated_text,
    publication_date="1920-05-10",
    filter_anachronistic=True,
)

print("Without date filtering:")
print(without_date_filter["summary"])
print("\nWith date filtering:")
print(with_date_filter["summary"])

The date is supplied by you; the pipeline does not infer it from the text. You can provide an integer year or a string containing a four-digit year. The filter uses the year only, not the exact month or day.

Sources are removed only when a supplied year is earlier than their known start year. If the publication year or source start year is missing, this filter cannot exclude that source. A historical date filter is a useful aid, but does not replace checking source histories and document metadata.

**Example 3:** Process several documents together

Pass a list of strings to get one result per document, in the same order. Here we use separate French, German and English examples. `batch_size` controls how many model chunks run together, rather than the number of documents returned.

In [ ]:
texts = [
    "Selon l'Agence Havas, les négociations se poursuivent à Paris.",
    "Die Deutsche Presse-Agentur (dpa) berichtet aus Berlin.",
    "Reuters reported from London. The BBC World Service broadcast the news.",
]

results = mediasources_pipeline(texts, batch_size=4)
for document_number, document_result in enumerate(results, start=1):
    print(f"Document {document_number}:", document_result["summary"])

For a collection with different publication dates, use `predict_many` and its plural `publication_dates` parameter. The dates must be in the same order as the texts, with one date per document. Use `None` for an unknown date.

In contrast, passing a single `publication_date` to a call on a list applies that date to every document.

In [ ]:
dated_results = mediasources_pipeline.predict_many(
    texts,
    publication_dates=["1898-05-14", "1955-06-01", "1920-05-10"],
    filter_anachronistic=True,
    batch_size=4,
)

for document_number, document_result in enumerate(dated_results, start=1):
    print(f"Document {document_number}:", document_result["summary"])

**Example 4:** `diagnostics`

Enable diagnostics to see the tokens and labels behind a prediction. `B-` begins a mention, `I-` continues it, and `O` marks a token outside a detected mention. Token scores are confidence values for the predicted labels.

In [ ]:
diagnostic_result = mediasources_pipeline(text, diagnostics=True)

for token, label, score in zip(
    diagnostic_result["tokens"],
    diagnostic_result["token_labels"],
    diagnostic_result["token_scores"],
):
    print(f"{token:<20} {label:<45} {score:.3f}")

The extra `token_start_offsets` and `token_end_offsets` fields locate each token in the original text. Diagnostics describe decoding before the optional mention filters; token labels can therefore still show a mention excluded by `min_score` or the date filter.

**Example 5:** Long documents

Long articles can exceed the model's input limit. The pipeline processes them in overlapping chunks, combines the predictions and retains character positions in the full original text.

We repeat a short sentence below to create a long input without downloading a dataset. This example demonstrates processing, rather than evaluating recognition quality.

In [ ]:
long_text = ("Reuters reported from London. The BBC World Service provided an update. " * 50).strip()
long_result = mediasources_pipeline(long_text, batch_size=2, diagnostics=True)

print("Mentions:", len(long_result["entities"]))
long_result["inference_diagnostics"]

`annotation_tokens` counts the text tokens and `model_chunks` shows how many primary chunks were processed. `overlapping_annotation_tokens` describes overlap between chunks; `overlap_replacements` counts tokens for which a later chunk supplied a more complete representation. `rescued_tokens` lists the positions of tokens the pipeline had to process separately because they received no prediction during chunk processing. An empty list means no tokens needed this extra step.

The pipeline also records aggregate batching and timing information from its **most recent call** in `last_inference_stats`. Larger batches can help on an accelerator but also use more memory. Reduce `batch_size` if you run out of memory.

In [ ]:
# Inspect these immediately after the long-document call.
mediasources_pipeline.last_inference_stats

**Example 6:** Combine the options

You can combine confidence filtering, date filtering and diagnostics in one call. This example uses the full sample text with a hypothetical 1920 publication date.

In [ ]:
mediasources_pipeline(
    text,
    min_score=0.90,
    publication_date=1920,
    filter_anachronistic=True,
    diagnostics=True,
)

### Model versions and offline use

The model and revision are selected when you initialize the pipeline, not when you annotate a text. Keeping the package version and model revision in your research notes makes results easier to reproduce.

Set `local_files_only=True` to load the model without downloading anything. This works only if all files for the exact model version (`v2.0.0` here) are already saved in the Hugging Face cache folder used by your current session. A model downloaded in Docker or Colab is not automatically available to a notebook running directly on your Mac. If files are missing, first initialize the pipeline with `local_files_only=False` while connected to the internet, then try offline mode again.

In [ ]:
# Optional: requires all files for v2.0.0 in this session's Hugging Face cache.
# offline_pipeline = MediaSourcesPipeline(
#     model="impresso-project/mmbert-impresso-mediasources-ner",
#     revision="v2.0.0",
#     local_files_only=True,
# )
# offline_pipeline(text)

## Limitations of the Media Sources Pipeline

- **Source coverage:** the current model recognizes press agencies and radio stations in its label set. It does not identify every organization or every possible source, such as arbitrary newspaper titles.
- **OCR and context:** misspellings, broken words and ambiguous abbreviations can cause missed mentions or wrong labels. Performance can differ between languages and collections.
- **Mention boundaries:** a prediction may cover only part of a source name. The character positions match the predicted span, which is not necessarily the full name a human would annotate.
- **Wikidata mappings:** some labels have no identifier yet. Treat `None` as a missing mapping, not as an unrecognized mention.
- **Historical filtering:** the date filter uses one saved start year for each source. However, an organization may have operated earlier under another name, and the model may identify that earlier name as the later organization. The filter could then remove a historically valid mention. Review these cases before discarding them from your analysis.
- **Scores:** high confidence does not guarantee a correct prediction or prove that the named source authored the text.

Try comparing clean and altered text. This is an illustrative OCR example, not a benchmark or a promise that the predictions will differ.

In [ ]:
ocr_examples = [
    "Reuters reported from London. The BBC World Service broadcast the news.",
    "Reutcrs reported from London. The B B C World Servlce broadcast the news.",
]

ocr_results = mediasources_pipeline(ocr_examples)
for example, example_result in zip(ocr_examples, ocr_results):
    print(example)
    print(example_result["entities"])
    print()

Empty input returns empty lists. Text with no recognized sources can also return empty lists; this does not prove that the document contains no source references.

In [ ]:
mediasources_pipeline("")

## Conclusion

In this notebook, we explored how to identify press agencies and radio stations with `MediaSourcesPipeline`. We inspected mention spans and source summaries, used confidence and publication-year filters, processed multiple documents, and examined diagnostics for long texts.

The pipeline provides a consistent way to annotate source mentions in your own collection. Inspect a sample of the results before using them for historical comparisons, and record the package version, model revision and filtering choices alongside your annotations.

## Next steps

- Explore the [OCR Quality Assessment demo](https://github.com/impresso/impresso-datalab-notebooks/blob/main/annotate/ocrqa_pipeline_demo.ipynb) for another view on your input text.
- Consult the [Media Sources pipeline documentation](https://github.com/impresso/impresso-pipelines/blob/main/README_mediasources.md) for configuration details.

---
## Project and License info

### Notebook credits [CreditLogo.png](https://credit.niso.org/)

Notebook contributor credits to be completed before publication.
<br></br>
This notebook is published under [CC BY 4.0 License](https://creativecommons.org/licenses/by/4.0/)
<br><a target="_blank" href="https://creativecommons.org/licenses/by/4.0/">
  <img src="https://mirrors.creativecommons.org/presskit/buttons/88x31/png/by.png"  width="100" alt="Open In Colab"/>
</a>
<br></br>

### Impresso project

[Impresso - Media Monitoring of the Past](https://impresso-project.ch) is an interdisciplinary research project that aims to develop and consolidate tools for processing and exploring large collections of media archives across modalities, time, languages and national borders. The first project (2017-2021) was funded by the Swiss National Science Foundation under grant No. [CRSII5_173719](http://p3.snf.ch/project-173719) and the second project (2023-2027) by the SNSF under grant No. [CRSII5_213585](https://data.snf.ch/grants/grant/213585) and the Luxembourg National Research Fund under grant No. 17498891.
<br></br>
### License

All Impresso code is published open source under the [GNU Affero General Public License](https://github.com/impresso/impresso-pyindexation/blob/master/LICENSE) v3 or later.


---

<p align="center">
  <img src="https://github.com/impresso/impresso.github.io/blob/master/assets/images/3x1--Yellow-Impresso-Black-on-White--transparent.png?raw=true" width="350" alt="Impresso Project Logo"/>
</p>